# Phase 4 · Step 2 — Prompt v2 (citation/grounding instructions)

Same question + same reranked chunks as notebook 08, but chunks are now
labeled with their page number in the prompt and the model is told to cite
pages for every claim. Compares v1 vs v2 side by side. See
`learning/06_prompts/02_prompt_v2.md` for the why behind this design.

## Reload, re-chunk, re-embed, upsert

Same setup as notebook 08 — rebuilds the hybrid-search collection from
scratch so this notebook is self-contained.

In [1]:
import logging

logging.getLogger("numexpr").setLevel(logging.WARNING)

from pathlib import Path

from generate_sample_pdf import generate
from llama_index.core.node_parser import SentenceSplitter
from llama_index.readers.file import PDFReader
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, Modifier, SparseVectorParams, VectorParams

COLLECTION_NAME = "papermind_chunks"
STORAGE_PATH = Path("data/qdrant")
DENSE_MODEL = "BAAI/bge-small-en-v1.5"
SPARSE_MODEL = "Qdrant/bm25"
RERANK_MODEL = "Xenova/ms-marco-MiniLM-L-6-v2"

pdf_path = generate()
documents = PDFReader().load_data(file=pdf_path)
splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

client = QdrantClient(path=str(STORAGE_PATH))
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)
client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

print(f"Produced {len(nodes)} chunks from {len(documents)} pages")

Produced 4 chunks from 4 pages


In [2]:
from uuid import NAMESPACE_URL, uuid5

from fastembed import SparseTextEmbedding, TextEmbedding
from qdrant_client.models import PointStruct, SparseVector

texts = [node.get_content() for node in nodes]

dense_embedder = TextEmbedding(model_name=DENSE_MODEL)
sparse_embedder = SparseTextEmbedding(model_name=SPARSE_MODEL)

dense_vectors = list(dense_embedder.embed(texts))
sparse_vectors = list(sparse_embedder.embed(texts))

page_counters: dict[str, int] = {}
points = []
for node, dense_vector, sparse_vector in zip(nodes, dense_vectors, sparse_vectors):
    page_label = node.metadata["page_label"]
    file_name = node.metadata["file_name"]
    chunk_index = page_counters.get(page_label, 0)
    page_counters[page_label] = chunk_index + 1

    point_id = str(uuid5(NAMESPACE_URL, f"{file_name}:{page_label}:{chunk_index}"))
    points.append(
        PointStruct(
            id=point_id,
            vector={
                "dense": dense_vector.tolist(),
                "bm25": SparseVector(
                    indices=sparse_vector.indices.tolist(),
                    values=sparse_vector.values.tolist(),
                ),
            },
            payload={
                "text": node.get_content(),
                "page_label": page_label,
                "file_name": file_name,
                "chunk_index": chunk_index,
            },
        )
    )

client.upsert(collection_name=COLLECTION_NAME, points=points)
print("Points count:", client.get_collection(COLLECTION_NAME).points_count)

Points count: 4


## Hybrid search + re-rank

Same as notebook 08: over-fetch 4 candidates with hybrid search, re-rank
with the cross-encoder, keep the top 2 as the final retrieved chunks.

In [3]:
import logging
import os

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

from fastembed.rerank.cross_encoder import TextCrossEncoder
from qdrant_client.models import FusionQuery, Fusion, Prefetch

question = "How is citation accuracy measured in this paper?"

query_dense = next(dense_embedder.embed([question])).tolist()
query_sparse = next(sparse_embedder.embed([question]))

hybrid_response = client.query_points(
    collection_name=COLLECTION_NAME,
    prefetch=[
        Prefetch(query=query_dense, using="dense", limit=10),
        Prefetch(
            query=SparseVector(
                indices=query_sparse.indices.tolist(),
                values=query_sparse.values.tolist(),
            ),
            using="bm25",
            limit=10,
        ),
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=4,
    with_payload=True,
)
candidates = hybrid_response.points

reranker = TextCrossEncoder(model_name=RERANK_MODEL)
candidate_texts = [c.payload["text"] for c in candidates]
rerank_scores = list(reranker.rerank(question, candidate_texts))
scored = sorted(zip(candidates, rerank_scores), key=lambda pair: pair[1], reverse=True)

TOP_K = 2
final_candidates = [c for c, _ in scored[:TOP_K]]
chunks = [c.payload for c in final_candidates]

for chunk in chunks:
    print(f"page {chunk['page_label']}: {chunk['text'][:60]}...")

page 1: Abstract
This paper studies retrieval-augmented generation f...
page 4: 3. Results
Across a held-out set of research papers, grounde...


## Answer with prompt v1 vs prompt v2

Needs `backend/.env` with `OPENAI_API_KEY` (set up in Phase 3). Two real,
billed API calls, same question + chunks, different prompt template.

In [4]:
import logging
import sys

logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

sys.path.insert(0, str(Path.cwd().parent))

from src.llm import get_llm
from src.prompts import v1, v2

llm = get_llm("openai")

print("=== v1 (no citation instructions) ===")
print(v1.answer(question, chunks, llm))
print()
print("=== v2 (citation instructions + page-labeled chunks) ===")
print("--- Prompt sent to the LLM ---")
print(v2.build_prompt(question, chunks))
print()
print("--- Answer ---")
print(v2.answer(question, chunks, llm))

=== v1 (no citation instructions) ===


2026-11-25 19:11:06,339 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context does not explicitly define how citation accuracy is measured. However, it implies that citation accuracy is evaluated based on the trustworthiness of grounded answers, which are answers that include citations to research papers. Human reviewers rated these grounded answers as more trustworthy compared to ungrounded answers, suggesting that citation accuracy may be assessed through the reviewers' evaluations of how well-cited answers are perceived in terms of reliability and credibility.

=== v2 (citation instructions + page-labeled chunks) ===
--- Prompt sent to the LLM ---
Context:
(page 1)
Abstract
This paper studies retrieval-augmented generation for question answering over research
papers. We show that combining dense retrieval with a re-ranking stage improves citation
accuracy compared to retrieval alone.

(page 4)
3. Results
Across a held-out set of research papers, grounded answers with page citations were rated
as more trustworthy than ungrounded answers by human re

2026-11-25 19:11:08,569 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The context does not explicitly detail how citation accuracy is measured in this paper. However, it does mention that combining dense retrieval with a re-ranking stage improves citation accuracy compared to retrieval alone (page 1). Additionally, it indicates that grounded answers with page citations were rated as more trustworthy than ungrounded answers by human reviewers (page 4), implying that trustworthiness may be a factor related to citation accuracy. The specific measurement method, however, is not provided in the context.


## Manual check

Comparing the two answers above by hand: does v2 actually cite page
numbers that v1 didn't? Does it cite the *right* pages? That observation
goes into the learning doc's "What we did".

## Next

Phase 4 · Step 3 adds a refuse-if-not-grounded rule (v3), then all three
versions (v1/v2/v3) get compared side by side on the same question.